In [160]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
import pickle
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Summarization with GPT5Nano

In [99]:
def extract_text_from_response(resp):
    texts = []

    for item in resp.output:
        # We only care about assistant messages
        if hasattr(item, "type") and item.type == "message":
            for content in item.content:
                if hasattr(content, "type") and content.type == "output_text":
                    texts.append(content.text)

    return "\n".join(texts).strip()

In [92]:
import re

def parse_sentiment_output(text):
    patterns = {
        "Impact": r"Impact:\s*(Bullish|Bearish|Neutral)",
        "ExpectedPctMove": r"Expected % Move:\s*([+-]?\d+)",
        "TimeHorizon": r"Time Horizon:\s*([^\n]+)",
        "Confidence": r"Confidence:\s*(\d+)",
        "Reason": r"Reason:\s*(.+)",
    }

    result = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        result[key] = match.group(1).strip() if match else None

    # Type coercion
    if result["ExpectedPctMove"] is not None:
        result["ExpectedPctMove"] = int(result["ExpectedPctMove"])

    if result["Confidence"] is not None:
        result["Confidence"] = int(result["Confidence"])

    return result

In [93]:
def summarize_filing(text):
    prompt = f"""
Summarize the following NSE corporate filing in 3–4 bullet points.

Focus on:
- What happened
- Quantitative details (amounts, capacity, dates)
- Impact on business or revenue

Ignore legal boilerplate, greetings, addresses.

FILING:
{text}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=300,
        reasoning={"effort": "minimal"},
    )

    return resp


In [94]:
def summarize_pulse_article(text):
    prompt = f"""
Summarize the following market news in 2–3 bullet points.

Focus on:
- The concrete event or trigger (if any)
- Which companies or sector are directly affected
- Ignore opinions, price targets, and generic market commentary

If there is no actionable event, say:
"Non-actionable market commentary."

NEWS:
{text}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=300,
        reasoning={"effort": "minimal"},
    )

    return resp


In [95]:
def classify_sentiment(summary):
    prompt = f"""
You are a market analyst evaluating the stock impact of a corporate event.

Based ONLY on the event summary below, provide a structured assessment.

Return the answer in the following format:

Impact: Bullish | Bearish | Neutral
Expected % Move: <single number, positive or negative, e.g. +4 or -3>
Time Horizon: <1–2 days | 3–5 days | 1–2 weeks>
Confidence: <integer from 0 to 100>
Reason: <one concise sentence explaining the impact>

Guidelines:
- Expected % Move should reflect a realistic short-term move for a liquid Indian F&O stock.
- Time Horizon refers to when most of the move is likely to materialize.
- Confidence reflects how certain the impact is, given the information quality and clarity.
- Do NOT invent facts or numbers not implied by the event.

EVENT SUMMARY:
{summary}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=400,
        reasoning={"effort": "low"},
    )

    return resp


In [138]:
with open("./Data/NSE/nse_news.pkl", "rb") as f:
    nse_news = pickle.load(f)
with open("./Data/Pulse/pulse_news.pkl", "rb") as f:
    pulse_news = pickle.load(f)

In [139]:
nse_news = nse_news[~nse_news["NEWS_EVENT"].isna()].reset_index(drop=True)
for i in tqdm(range(len(nse_news))):

    article = nse_news.loc[i, "NEWS_EVENT"]

    # --- Summarization ---
    summary_resp = summarize_pulse_article(article)
    summary_text = extract_text_from_response(summary_resp)

    # --- Sentiment / impact ---
    sentiment_resp = classify_sentiment(summary_text)
    sentiment_text = extract_text_from_response(sentiment_resp)

    parsed = parse_sentiment_output(sentiment_text)

    # --- Write to dataframe ---
    nse_news.loc[i, "Summary"] = summary_text
    nse_news.loc[i, "Impact"] = parsed["Impact"]
    nse_news.loc[i, "ExpectedPctMove"] = parsed["ExpectedPctMove"]
    nse_news.loc[i, "TimeHorizon"] = parsed["TimeHorizon"]
    nse_news.loc[i, "Confidence"] = parsed["Confidence"]
    nse_news.loc[i, "Reason"] = parsed["Reason"]

100%|██████████| 14/14 [01:03<00:00,  4.56s/it]


In [146]:
for i in tqdm(range(len(pulse_news))):

    article = pulse_news.loc[i, "Complete_Article"]

    # --- Summarization ---
    summary_resp = summarize_pulse_article(article)
    summary_text = extract_text_from_response(summary_resp)

    # --- Sentiment / impact ---
    sentiment_resp = classify_sentiment(summary_text)
    sentiment_text = extract_text_from_response(sentiment_resp)

    parsed = parse_sentiment_output(sentiment_text)

    # --- Write to dataframe ---
    pulse_news.loc[i, "Summary"] = summary_text
    pulse_news.loc[i, "Impact"] = parsed["Impact"]
    pulse_news.loc[i, "ExpectedPctMove"] = parsed["ExpectedPctMove"]
    pulse_news.loc[i, "TimeHorizon"] = parsed["TimeHorizon"]
    pulse_news.loc[i, "Confidence"] = parsed["Confidence"]
    pulse_news.loc[i, "Reason"] = parsed["Reason"]


100%|██████████| 34/34 [02:47<00:00,  4.94s/it]


In [164]:
nse_news_final = nse_news[nse_news['Summary']!='Non-actionable market commentary.']
nse_news_final = nse_news_final[nse_news_final['Impact']!='Neutral']
nse_news_final['RECEIPT'] = pd.to_datetime(nse_news_final['RECEIPT']) 
nse_news_final = nse_news_final[nse_news_final['Confidence']>60].sort_values(by="Confidence", ascending=False)

nse_news_final = nse_news_final[['RECEIPT','SYMBOL','Impact','ExpectedPctMove','Confidence','Reason','DETAILS']].rename(columns={'RECEIPT':'Date', 'DETAILS':'Description', 'SYMBOL':'Tags'})

In [165]:
pulse_news_final = pulse_news[pulse_news['Summary']!='Non-actionable market commentary.']
pulse_news_final = pulse_news_final[pulse_news_final['Impact']!='Neutral']
pulse_news_final['Date'] = pd.to_datetime(pulse_news_final['Date'])
pulse_news_final = pulse_news_final[pulse_news_final['Confidence']>60].sort_values(by="Confidence", ascending=False)

pulse_news_final = pulse_news_final[['Date','Tags','Impact','ExpectedPctMove','Confidence','Reason','Description']]

In [166]:
# -----------------------------
# 1. Combine latest run
# -----------------------------
final_df = pd.concat(
    [nse_news_final, pulse_news_final],
    ignore_index=True
)

# -----------------------------
# 2. Month-based folder
# -----------------------------
month_str = datetime.now().strftime("%Y-%m")   # e.g. 2026-01
base_dir = Path("./Data/News_Analysis")
month_dir = base_dir / month_str
month_dir.mkdir(parents=True, exist_ok=True)

csv_path = month_dir / "news_analysis.csv"

# -----------------------------
# 3. Append to existing data
# -----------------------------
if csv_path.exists():
    historical_df = pd.read_csv(csv_path)
    historical_df["Date"] = pd.to_datetime(historical_df["Date"])

    combined_df = pd.concat([historical_df, final_df],ignore_index=True)
else:
    combined_df = final_df.copy()

# -----------------------------
# 4. Optional: de-duplicate
# (highly recommended)
# -----------------------------
# Pick stable identifiers you trust
dedupe_cols = ['Date','Tags','Description']

combined_df = (combined_df.drop_duplicates(subset=dedupe_cols, keep="last"))

# -----------------------------
# 5. Sort latest first
# -----------------------------
combined_df = combined_df.sort_values(by="Date",ascending=False).reset_index(drop=True)

# -----------------------------
# 6. Save back
# -----------------------------
combined_df.to_csv(csv_path, index=False)

print(f"Saved {len(combined_df)} rows to {csv_path}")

Saved 14 rows to Data/News_Analysis/2026-01/news_analysis.csv


In [ ]:
#TODO: 1. Publish via Telegram
#TODO: 2. Use kiteconnect to check results of these predictions